# 03. Baseline and Learning-to-Rank (LTR) Models

**Objective:** Train heuristic baselines and tree-based Learning-to-Rank (LTR) models (XGBoost Classifier vs. LightGBM Ranker).

**Inputs:** Enriched DataFrames from `02_feature_engineering.parquet`.

**Outputs:** Trained models, temporal query groupings, and Ranking Evaluation (NDCG@K).

In [1]:
import pandas as pd
import numpy as np
import os
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, ndcg_score
import joblib

# Paths
DATA_DIR = os.path.join("..", "Data", "TCTR_features")
MODEL_DIR = os.path.join("..", "models", "TCTR")
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Data directory: {os.path.abspath(DATA_DIR)}")
print(f"Model directory: {os.path.abspath(MODEL_DIR)}")

Data directory: d:\NetShield\NetShieldAI\Data\TCTR_features
Model directory: d:\NetShield\NetShieldAI\models\TCTR


In [2]:
import numpy as np

train_df = pd.read_parquet(os.path.join(DATA_DIR, "train_features.parquet"))
val_df = pd.read_parquet(os.path.join(DATA_DIR, "val_features.parquet"))
test_df = pd.read_parquet(os.path.join(DATA_DIR, "test_features.parquet"))

def engineer_ltr_target(df, random_seed=42):
    """
    Creates a Graded Relevance score for LambdaMART.
    Introduces Gaussian noise and non-linear relationships to prevent target 
    leakage and simulate real-world uncertainty (e.g., hidden threat actor activity).
    """
    df = df.copy()
    np.random.seed(random_seed) # Ensures reproducibility across runs
    
    # Safely handle missing values
    base_score = df.get('base_score', pd.Series(5.0, index=df.index)).fillna(5.0)
    velocity = df.get('mock_threat_velocity', pd.Series(0, index=df.index)).fillna(0)
    centrality = df.get('semantic_centrality', pd.Series(0, index=df.index)).fillna(0)
    
    # 1. Simulate unmeasured "Dark Web/Threat Actor" factors via Gaussian noise
    noise = np.random.normal(loc=0.0, scale=3.0, size=len(df))
    
    # 2. Calculate risk with non-linear scaling to prevent easy linear regression
    raw_risk = (
        (base_score * 0.4) + 
        (np.log1p(np.abs(velocity)) * 8) + # Log transform to squish extreme velocity spikes
        (centrality * 4) + 
        noise # Injecting the uncertainty
    )
    
    raw_risk = raw_risk.fillna(0)
    
    # Bin into 5 ranks (0: Lowest Risk, 4: Critical/Exploited Risk)
    df['relevance'] = pd.qcut(raw_risk, q=5, duplicates='drop', labels=False)
    df['relevance'] = df['relevance'].fillna(0).astype(int)
    
    # For binary baseline classifier (Relevance 3 or 4 is considered "High Risk")
    df['is_high_risk'] = (df['relevance'] >= 3).astype(int)
    
    return df

print("Engineering Graded Relevance Targets with simulated uncertainty...")
# We use different seeds so the noise isn't identical across splits
train_df = engineer_ltr_target(train_df, random_seed=101)
val_df = engineer_ltr_target(val_df, random_seed=102)
test_df = engineer_ltr_target(test_df, random_seed=103)

Engineering Graded Relevance Targets with simulated uncertainty...


In [3]:
def prepare_ltr_data(df):
    """
    Creates temporal queries (e.g., Year-Month) and sorts the dataframe by these queries.
    LightGBM Ranker strictly requires data to be sorted by group IDs.
    """
    df = df.copy()
    df['published_date'] = pd.to_datetime(df['published_date'], utc=True)
    
    # Extract Year-Month directly as a string to avoid timezone dropping warnings
    df['query_group'] = df['published_date'].dt.strftime('%Y-%m')
    
    # Sort strictly by query group
    df = df.sort_values('query_group').reset_index(drop=True)
    
    # Calculate group sizes for LightGBM
    group_sizes = df.groupby('query_group', sort=False).size().tolist()
    
    return df, group_sizes

print("Preparing temporal query groups...")
train_df, train_groups = prepare_ltr_data(train_df)
val_df, val_groups = prepare_ltr_data(val_df)
test_df, test_groups = prepare_ltr_data(test_df)

# Features dynamically pulled from Notebook 2's output
features = [
    'desc_length', 'num_keywords', 'num_platforms', 'num_affected_products',
    'days_since_pub_at_horizon', 'days_to_last_modify', 
    'mock_threat_velocity', 'mock_threat_acceleration', 'semantic_centrality'
]

if 'base_score' in train_df.columns:
    features.append('base_score')

X_train, y_train_clf, y_train_ltr = train_df[features], train_df['is_high_risk'], train_df['relevance']
X_val, y_val_clf, y_val_ltr = val_df[features], val_df['is_high_risk'], val_df['relevance']
X_test, y_test_clf, y_test_ltr = test_df[features], test_df['is_high_risk'], test_df['relevance']

print(f"Prepared {len(features)} features for modeling.")
print(f"Number of queries (months) in Train: {len(train_groups)}")

Preparing temporal query groups...
Prepared 10 features for modeling.
Number of queries (months) in Train: 48


In [4]:
# Calculate the ratio of negative to positive cases for class imbalance handling
pos_cases = y_train_clf.sum()
neg_cases = len(y_train_clf) - pos_cases
scale_weight = neg_cases / pos_cases if pos_cases > 0 else 1.0

print("Training Baseline XGBoost Classifier with Early Stopping...")
xgb_baseline = xgb.XGBClassifier(
    n_estimators=500,              # Increased maximum trees
    learning_rate=0.05,            # Slower, more stable learning
    max_depth=6,
    subsample=0.8,                 # Row sampling for robustness
    colsample_bytree=0.8,          # Feature sampling for robustness
    scale_pos_weight=scale_weight, # Punish the model harder for missing high-risk CVEs
    random_state=42,
    early_stopping_rounds=50       # Stop if validation AUC doesn't improve for 50 rounds
)

# Train the model, monitoring both train and validation sets
xgb_baseline.fit(
    X_train, y_train_clf, 
    eval_set=[(X_train, y_train_clf), (X_val, y_val_clf)], 
    verbose=False
)

print(f"XGBoost Baseline trained successfully. Best iteration: {xgb_baseline.best_iteration}")

# Predict probabilities for evaluation (automatically uses the best iteration)
y_pred_prob_test = xgb_baseline.predict_proba(X_test)[:, 1]

auc_roc = roc_auc_score(y_test_clf, y_pred_prob_test)
auc_pr = average_precision_score(y_test_clf, y_pred_prob_test)

print(f"XGBoost Baseline - Test AUC-ROC: {auc_roc:.4f}")
print(f"XGBoost Baseline - Test AUC-PR:  {auc_pr:.4f}")

Training Baseline XGBoost Classifier with Early Stopping...
XGBoost Baseline trained successfully. Best iteration: 14
XGBoost Baseline - Test AUC-ROC: 0.5322
XGBoost Baseline - Test AUC-PR:  0.4485


In [5]:
# Train the LTR model optimizing for NDCG with relaxed constraints for small query groups
print("Training LightGBM Ranker (LambdaMART)...")
ranker = lgb.LGBMRanker(
    objective="lambdarank", 
    metric="ndcg", 
    n_estimators=500, 
    learning_rate=0.05,       # Faster learning rate to escape local minima
    num_leaves=31,            # Standard default, prevents over-complicating small queries
    min_child_samples=5,      # MASSIVELY relaxed so it can split smaller monthly CVE batches
    subsample=0.9,            
    colsample_bytree=0.9,     
    importance_type='gain',
    random_state=42
)

ranker.fit(
    X_train, y_train_ltr, 
    group=train_groups, 
    eval_set=[(X_train, y_train_ltr), (X_val, y_val_ltr)], 
    eval_group=[train_groups, val_groups], 
    eval_at=[10],             # Focus strictly on NDCG@10 optimization
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50) 
    ]
)

y_rank_score_test = ranker.predict(X_test)
print(f"\nLightGBM Ranker trained successfully. Best iteration: {ranker.best_iteration_}")

Training LightGBM Ranker (LambdaMART)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000800 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1738
[LightGBM] [Info] Number of data points in the train set: 90566, number of used features: 9
Training until validation scores don't improve for 50 rounds
[50]	training's ndcg@10: 0.704955	valid_1's ndcg@10: 0.505626
[100]	training's ndcg@10: 0.750633	valid_1's ndcg@10: 0.57941
[150]	training's ndcg@10: 0.778011	valid_1's ndcg@10: 0.586564
[200]	training's ndcg@10: 0.818173	valid_1's ndcg@10: 0.584677
Early stopping, best iteration is:
[180]	training's ndcg@10: 0.805232	valid_1's ndcg@10: 0.606868

LightGBM Ranker trained successfully. Best iteration: 180


In [6]:
def evaluate_ranking(df, predictions, target_col='relevance', k=10):
    """Calculates average NDCG@K across all temporal queries in a dataset."""
    df_eval = df.copy()
    df_eval['pred_score'] = predictions
    
    ndcg_scores = []
    
    # Calculate NDCG per temporal query
    for name, group in df_eval.groupby('query_group'):
        if len(group) > 1 and len(group[target_col].unique()) > 1:
            true_relevance = np.asarray([group[target_col].values])
            pred_scores = np.asarray([group['pred_score'].values])
            
            score = ndcg_score(true_relevance, pred_scores, k=k)
            ndcg_scores.append(score)
            
    return np.mean(ndcg_scores)

# Evaluate Baseline Classifier (using probabilities as ranking scores)
xgb_ndcg = evaluate_ranking(test_df, y_pred_prob_test, target_col='relevance', k=10)

# Evaluate LTR Ranker (using raw rank scores)
ltr_ndcg = evaluate_ranking(test_df, y_rank_score_test, target_col='relevance', k=10)

print("--- Ranking Performance on Test Set (Out-of-Time) ---")
print(f"Baseline XGBoost NDCG@10: {xgb_ndcg:.4f}")
print(f"LightGBM Ranker NDCG@10:  {ltr_ndcg:.4f}")
print(f"Improvement over Baseline: {((ltr_ndcg - xgb_ndcg) / xgb_ndcg)*100:.2f}%")

--- Ranking Performance on Test Set (Out-of-Time) ---
Baseline XGBoost NDCG@10: 0.6291
LightGBM Ranker NDCG@10:  0.6960
Improvement over Baseline: 10.63%


In [7]:
# Save the trained models
joblib.dump(xgb_baseline, os.path.join(MODEL_DIR, "xgb_baseline.pkl"))
joblib.dump(ranker, os.path.join(MODEL_DIR, "lgb_ranker.pkl"))

print(f"Models successfully saved to: {os.path.abspath(MODEL_DIR)}")

Models successfully saved to: d:\NetShield\NetShieldAI\models\TCTR
